In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
KEY = os.environ.get("RIOT_API_KEY") 

if KEY is None:
    print("Error: API_KEY environment variable not set.")



In [2]:
import pandas as pd
import itertools
import csv

d_puuids = []
filename = "puuids2.csv"

with open(filename, 'r', newline='', encoding='utf-8') as csvfile:
    csv_reader = csv.reader(csvfile, delimiter=',')
    for row in csv_reader:
        d_puuids.append(row)

puuids = list(itertools.chain.from_iterable(d_puuids))


In [ ]:
import requests

def get_match_history_ids(puuid, api_key, region, start=0, count=2):
    api_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?start={start}&count={count}&api_key={api_key}"
    response = requests.get(api_url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching match IDs: {response.status_code} - {response.text}")
        return None
    
def get_match_details(match_id, api_key, region):
    api_url = f"https://{region}.api.riotgames.com/lol/match/v5/matches/{match_id}?api_key={api_key}"
    response = requests.get(api_url)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Error fetching match details for {match_id}: {response.status_code} - {response.text}")
        return None

In [7]:
player_stats = next(p for p in data["info"]["participants"] if p["puuid"] == puuids[0])
len(player_stats)

147

In [8]:
features = [
    "championId", "teamId",
    "kills", "deaths", "assists",
    "totalDamageDealtToChampions", "totalDamageTaken",
    "goldEarned", "goldSpent",
    "totalMinionsKilled", "neutralMinionsKilled",
    "visionScore", "wardsPlaced", "wardsKilled", "detectorWardsPlaced",
    "dragonKills", "baronKills", "turretKills", "inhibitorKills",
    "firstBloodKill", "firstTowerKill",
    "totalHeal", "totalHealsOnTeammates",
    "damageSelfMitigated",
    "totalTimeCCDealt", "timeCCingOthers",
    "item0", "item1", "item2", "item3", "item4", "item5", "item6",
    "summoner1Id", "summoner2Id",
    "win", "gameEndedInEarlySurrender", "gameEndedInSurrender",
    "timePlayed",
    "spell1Casts", "spell2Casts", "spell3Casts", "spell4Casts",
]

def get_features(puuid):
    stats = {field: puuid.get(field) for field in features}
    return stats



In [9]:
from datetime import datetime

timestamp = data["info"]["gameStartTimestamp"]
game_start = datetime.fromtimestamp(timestamp / 1000)

player_feats = get_features(player_stats)
player_feats['game_start'] = game_start

player_feats

{'championId': 67,
 'teamId': 200,
 'kills': 0,
 'deaths': 7,
 'assists': 1,
 'totalDamageDealtToChampions': 12254,
 'totalDamageTaken': 17654,
 'goldEarned': 5168,
 'goldSpent': 5000,
 'totalMinionsKilled': 109,
 'neutralMinionsKilled': 0,
 'visionScore': 19,
 'wardsPlaced': 7,
 'wardsKilled': 3,
 'detectorWardsPlaced': 0,
 'dragonKills': 0,
 'baronKills': 0,
 'turretKills': 0,
 'inhibitorKills': 0,
 'firstBloodKill': False,
 'firstTowerKill': False,
 'totalHeal': 2805,
 'totalHealsOnTeammates': 0,
 'damageSelfMitigated': 6025,
 'totalTimeCCDealt': 11,
 'timeCCingOthers': 6,
 'item0': 1055,
 'item1': 3006,
 'item2': 2031,
 'item3': 3073,
 'item4': 2022,
 'item5': 0,
 'item6': 3340,
 'summoner1Id': 4,
 'summoner2Id': 6,
 'win': False,
 'gameEndedInEarlySurrender': False,
 'gameEndedInSurrender': True,
 'timePlayed': 1160,
 'spell1Casts': 93,
 'spell2Casts': 0,
 'spell3Casts': 11,
 'spell4Casts': 5,
 'game_start': datetime.datetime(2026, 4, 13, 0, 10, 59, 43000)}

In [10]:
import time
time_series = {}
count = 1
time.sleep(120)
for puuid in puuids:
    print(puuid)
    match_ids = get_match_history_ids(puuid, KEY, "americas", 0, 90)
    player_feats = {}
    if match_ids is not None:
        all_matches = []
        for match in match_ids:
            details = get_match_details(match, KEY, "americas")
            if details is not None:
                try:
                    player_stats = next(p for p in details["info"]["participants"] if p["puuid"] == puuid)
                except:
                    print("failed retrieving player_stats")
                    break
                timestamp = details["info"]["gameStartTimestamp"]
                game_start = datetime.fromtimestamp(timestamp / 1000)

                player_feats = get_features(player_stats)
                player_feats['game_start'] = game_start
                all_matches.append(player_feats)
        time_series[puuid] = all_matches

        time.sleep(120)
        count+=1

q9wnSUw7ft1BLkMuoHA_xBSRbPYBGzMzdWLStuPm5TH9Afd-ov8ZXnjxkqSKQTiVfXVai8DKemyU5w
8RO5RrBGD4laxG_e7ZDzV4z1hoFWcp0U71D2PKZl04YyCtPN3FvEJbK7cneXUpcrnlBP8A4ZWfzrug
xPeCEi50IIAEWXniRZVTyQu-1vG2paFq5v6NKKziF4I6Sqk9B8xG8lGkPDzAFU3GqyL-CAM522S6ZA
rHaQBwLZXKz7fGslPq6UzTRR_1RbQ0H1Gtu0CdCM79SqePj_6FkI6Akn2pUSBrj3cAIvKnrZMXSPVQ
7Fs_0z76dhJ7ji8lVhCRWMVK5t5C6UYxWSCMRVr3ucvhcuuhLcgHH7tHNY7e65ck5v9gs4Y0fUlhbg
3n6X1SbFpKxY8UPXQyFGGiHuxWXiIb61-gskS1vSYwsvP35o_93ugUarUkImX7Eu2-dmXAmfflN2Fg
DJ7J-w-bwJHRk3_8jO8u2MSST9gDIR_VvyqYfccSp934RW1fm9Y-eTbL227UfSbapqpBPpeDZY3ZZg
vixSdaEFraow_BreXYpYJ26klTsvwNg-GhszZfzEe-DKNsH3qwqF2sx6VS5mXK31qi4sKPiUMZJIsw
28_bMw64gU0IJX4Cb5RNPbklbcXGHjYtRsbDk8POBnfQ7uUNugucKpcrqvjr-5nxd9kZgq9Rn8F8Gw
Vj38p7eOaIhW_DxdidVcVmGYMTRhBEw5ZfDcU-QA1bfL4AfNiW3NNIYGksQ8ZkafVN--BZuwLXDpAQ
JS-IBQuqZ-K90ckLxX-00I1dFalkwu6hPQrPYJMd-m7qBPtqaUgEQg5WEThGQEwPo11thgcBshJuoA
YXI3ZyinUEYJUwC7JF4Jcb6o07-0RlM8V_lAtbUyFfK5EUTt7oYTgcmiGJDbmLXlolpt2HEGW3Gf7g
nMxYtcDVZeMqLvQBvI94Kkv7EvkeGSQi78M_VfZayv5BKnJtuUsi

KeyboardInterrupt: 

In [13]:
len(time_series)

133

In [12]:
import csv

with open("match_data2.csv", "w", newline="") as f:
    writer = csv.writer(f)

    for key, values in time_series.items():
        writer.writerow([key] + values)